## 1. Import Library and Setup Configuration

In [11]:
# import necessary libraries
import os
import sys
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score
from health_multimodal.image import get_image_inference
from health_multimodal.image.utils import ImageModelType

In [12]:
# setup reproducibility
def seed_everything(seed=42):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [3]:
# setup configuration
config = {
    "experiment": "exp_001",
    "stage": "stage_b",

    "data": {
        "train_path": r"D:\VLM-Research_Task-C\data\processed\chexpert_plus\train.parquet",
        "dev_path": r"D:\VLM-Research_Task-C\data\processed\chexpert_plus\dev_internal.parquet",
        "test_path": r"D:\VLM-Research_Task-C\data\processed\chexpert_plus\test.parquet",
        "image_col": "actual_image_path",
        "label_suffix": "_label"
    },
    
    "model": {
        "freeze_backbone": False,
        "hidden_dim": None,
        "num_pathologies": 14,
        "dropout": 0.1
    },
    
    "train": {
        "batch_size": 16,
        "num_epochs": 30,
        "lr": 1.0e-5,
        "weight_decay": 1.0e-4,
        "grad_clip_norm": 1.0,
        "early_stopping_patience": 7,
        "num_workers": 0,
        "seed": 42,
        "lr_scheduler": "ReduceLROnPlateau",
        "lr_scheduler_patience": 3
    },
    
    "output": {
        "checkpoint_dir": r"D:\VLM-Research_Task-C\output\exp_001\stage_b\checkpoints",
        "log_dir": r"D:\VLM-Research_Task-C\output\exp_001\stage_b\logs",
        "tuning_dir": r"D:\VLM-Research_Task-C\output\exp_001\stage_b\tuning",
        "plot_dir": r"D:\VLM-Research_Task-C\output\exp_001\stage_b\plots",
        "best_model_filename": "stage_b_best.pt"
    },
    
    "device": "cuda"
}

# create output directories
for key in ["checkpoint_dir", "log_dir", "tuning_dir", "plot_dir"]:
    os.makedirs(config["output"][key], exist_ok=True)

In [4]:
# define device, if GPU doesn't exist, it'll fallback to CPU usage
device = torch.device(config['device'] if torch.cuda.is_available() else 'cpu')
print(f"using device: {device}")

using device: cuda


## 2. Load Data

In [5]:
# load into a dataframe for each set
df_train = pd.read_parquet(config['data']['train_path'])
df_dev = pd.read_parquet(config['data']['dev_path'])
df_test = pd.read_parquet(config['data']['test_path'])

In [6]:
# quick exploration

# dataset dimension
print(f"train set: {df_train.shape[0]} rows & {df_train.shape[1]} columns")
print(f"dev set: {df_dev.shape[0]} rows & {df_dev.shape[1]} columns")
print(f"test set: {df_test.shape[0]} rows & {df_test.shape[1]} columns")

# CXR pathology multilabel
label_cols = [col for col in df_train.columns if col.endswith(config['data']['label_suffix'])]
print(f"label columns ({len(label_cols)}): {label_cols}")

# make sure there are 14 CXR pathology labels
assert len(label_cols) == 14, "there have to be 14 CXR pathology labels"

train set: 189529 rows & 77 columns
dev set: 33563 rows & 77 columns
test set: 234 rows & 77 columns
label columns (14): ['Enlarged Cardiomediastinum_label', 'Cardiomegaly_label', 'Lung Opacity_label', 'Lung Lesion_label', 'Edema_label', 'Consolidation_label', 'Pneumonia_label', 'Atelectasis_label', 'Pneumothorax_label', 'Pleural Effusion_label', 'Pleural Other_label', 'Fracture_label', 'Support Devices_label', 'No Finding_label']


## 3. Training Utilities

### 3.1 Dataset Class

In [7]:
class CheXpertPlusDataset(Dataset):
    def __init__(self, df, image_col, label_cols, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_cols = label_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row[self.image_col]
        try:
            image = Image.open(img_path).convert('L')
        except FileNotFoundError:
            print(f"warning: image not found {img_path}, returning zeros.")
            image = Image.new('L', (224, 224))
        if self.transform:
            image = self.transform(image)
        labels = torch.tensor(row[self.label_cols].values.astype(np.float32), dtype=torch.float32)
        return image, labels

### 3.2 Load BioViL-T Image Encoder

In [8]:
# load BioViL-T image encoder
image_engine = get_image_inference(ImageModelType.BIOVIL_T)
biovilt_transform = image_engine.transform
bio_model = image_engine.model
bio_model.to(device)
bio_model.eval()

print("biovil-t loaded successfully")

biovil-t loaded successfully


In [13]:
print("bio_model top-level children:")
for name, _ in bio_model.named_children():
    print(f"  {name}")

print("\nbio_model.encoder children:")
for name, _ in bio_model.encoder.named_children():
    print(f"  {name}")

print("\nbio_model.encoder.encoder (resnet) children:")
for name, _ in bio_model.encoder.encoder.named_children():
    print(f"  {name}")

bio_model top-level children:
  encoder
  projector

bio_model.encoder children:
  encoder
  backbone_to_vit
  vit_pooler

bio_model.encoder.encoder (resnet) children:
  conv1
  bn1
  relu
  maxpool
  layer1
  layer2
  layer3
  layer4
  avgpool
  fc


In [10]:
print(bio_model)

ImageModel(
  (encoder): MultiImageEncoder(
    (encoder): ResNetHIML(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (rel

In [9]:
# detect vision hidden dimension

sample_row = df_train.iloc[0]
sample_img_path = sample_row[config['data']['image_col']]
try:
    sample_img = Image.open(sample_img_path).convert('L')
except FileNotFoundError:
    sample_img = Image.new('L', (224, 224))

sample_pixel_values = biovilt_transform(sample_img).unsqueeze(0).to(device)

with torch.no_grad():
    dummy_out = bio_model(sample_pixel_values)
    
    # extract features tensor from dummy_out
    if hasattr(dummy_out, 'patch_embeddings'):
        features = dummy_out.patch_embeddings
        print("using patch_embeddings")
    elif hasattr(dummy_out, 'last_hidden_state'):
        features = dummy_out.last_hidden_state
        print("using last_hidden_state")
    else:
        # if dummy_out is a tensor directly, use it
        if isinstance(dummy_out, torch.Tensor):
            features = dummy_out
            print("using raw tensor output")
        else:
            # fallback: try to get encoder output
            try:
                features = bio_model.encoder(sample_pixel_values)
                print("using encoder output")
            except:
                raise RuntimeError("Cannot extract features from BioViL-T output")
    
    # determine vision dimension based on features shape
    if features.dim() == 3:
        # (B, seq, D)
        vision_dim = features.shape[-1]
    elif features.dim() == 4:
        # (B, C, H, W)
        vision_dim = features.shape[1]
    else:
        # fallback
        vision_dim = features.shape[-1]
    
    # sanity check: if vision_dim is suspiciously small (like 14), set to a known good value
    if vision_dim < 100:
        print(f"warning: detected vision_dim={vision_dim}, which is suspiciously small. forcing to 768 (default BioViL-T dimension).")
        vision_dim = 768

config['vision_dim'] = vision_dim
print(f"vision hidden dimension: {vision_dim}")

using patch_embeddings
vision hidden dimension: 512


### 3.3 Create Dataset and Dataloader

In [10]:
train_dataset = CheXpertPlusDataset(df_train, config['data']['image_col'], label_cols, transform=biovilt_transform)
dev_dataset = CheXpertPlusDataset(df_dev, config['data']['image_col'], label_cols, transform=biovilt_transform)
test_dataset = CheXpertPlusDataset(df_test, config['data']['image_col'], label_cols, transform=biovilt_transform)

train_loader = DataLoader(train_dataset, batch_size=config['train']['batch_size'], shuffle=True, num_workers=config['train']['num_workers'], pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=config['train']['batch_size'], shuffle=False, num_workers=config['train']['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config['train']['batch_size'], shuffle=False, num_workers=config['train']['num_workers'], pin_memory=True)

print(f"train batches: {len(train_loader)}, dev batches: {len(dev_loader)}, test batches: {len(test_loader)}")

train batches: 23692, dev batches: 4196, test batches: 30


### 3.4 Compute Weights to Mitigate Imbalanced Class

In [11]:
def compute_pos_weights(df: pd.DataFrame, label_cols: list) -> torch.Tensor:
    """
    per-pathology pos_weight for nn.BCEWithLogitsLoss, from train prevalence only.
    pos_weight_i = num_negative_i / num_positive_i.
    """
    pos_counts = (df[label_cols] == 1.0).sum()
    neg_counts = (df[label_cols] == 0.0).sum()
    pos_weight = (neg_counts / pos_counts.clip(lower=1)).to_numpy(dtype="float32")
    return torch.tensor(pos_weight)

pos_weight = compute_pos_weights(df_train, label_cols).to(device)

print("class weights (pos_weight = N_neg / N_pos):")
for col, w in zip(label_cols, pos_weight.cpu().numpy()):
    print(f"  {col}: {w:.3f}")

class weights (pos_weight = N_neg / N_pos):
  Enlarged Cardiomediastinum_label: 5.850
  Cardiomegaly_label: 4.359
  Lung Opacity_label: 0.958
  Lung Lesion_label: 16.982
  Edema_label: 2.359
  Consolidation_label: 4.224
  Pneumonia_label: 7.928
  Atelectasis_label: 1.999
  Pneumothorax_label: 9.801
  Pleural Effusion_label: 1.179
  Pleural Other_label: 26.580
  Fracture_label: 19.382
  Support Devices_label: 0.697
  No Finding_label: 12.279


### 3.5 Define Stage-B Classifier Model

In [12]:
class StageBClassifier(nn.Module):
    def __init__(self, config, backbone, vision_dim, freeze_backbone=False):
        super().__init__()
        self.backbone = backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        self.hidden_dim = config['model']['hidden_dim']
        self.num_classes = config['model']['num_pathologies']
        self.dropout_rate = config['model']['dropout']
        
        in_features = vision_dim
        
        if self.hidden_dim is not None:
            self.classifier = nn.Sequential(
                nn.Dropout(self.dropout_rate),
                nn.Linear(in_features, self.hidden_dim),
                nn.ReLU(),
                nn.Dropout(self.dropout_rate),
                nn.Linear(self.hidden_dim, self.num_classes)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Dropout(self.dropout_rate),
                nn.Linear(in_features, self.num_classes)
            )
    
    def forward(self, x):
        out = self.backbone(x)
        
        # extract features
        if hasattr(out, 'patch_embeddings'):
            features = out.patch_embeddings
        elif hasattr(out, 'last_hidden_state'):
            features = out.last_hidden_state
        else:
            features = out
        
        # ensure features is a tensor
        if not isinstance(features, torch.Tensor):
            raise TypeError(f"Expected Tensor, got {type(features)}")
        
        # pooling: reduce to (B, D)
        if features.dim() == 3:
            pooled = features.mean(dim=1)  # (B, seq, D) -> (B, D)
        elif features.dim() == 4:
            pooled = features.mean(dim=[2, 3])  # (B, C, H, W) -> (B, C)
        else:
            # already pooled? just flatten if needed
            pooled = features.flatten(1) if features.dim() > 2 else features
        
        logits = self.classifier(pooled)
        return logits

# instantiate model
model = StageBClassifier(
    config,
    backbone=bio_model,
    vision_dim=config['vision_dim'],
    freeze_backbone=config['model']['freeze_backbone']
)
model = model.to(device)

if config['model']['freeze_backbone']:
    for param in bio_model.parameters():
        param.requires_grad = False

print(f"model initialized on {device}")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total parameters: {total_params/1e6:.2f}M, trainable: {trainable_params/1e6:.2f}M")

model initialized on cuda
total parameters: 27.36M, trainable: 27.36M


## 4. Train Loop

In [13]:
no_finding_idx = None
for i, col in enumerate(label_cols):
    if col.lower().startswith('no finding'):
        no_finding_idx = i
        break
if no_finding_idx is None:
    no_finding_idx = 0
other_indices = [i for i in range(len(label_cols)) if i != no_finding_idx]

print(f"no finding index: {no_finding_idx}")
print(f"other indices (13 pathologies): {other_indices}")

no finding index: 13
other indices (13 pathologies): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [14]:
torch.cuda.empty_cache()

In [15]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['train']['lr'],
    weight_decay=config['train']['weight_decay']
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=config['train']['lr_scheduler_patience'], factor=0.5
)

# history tracking
early_stop_counter = 0
best_val_macro_f1 = 0.0
train_losses, val_losses = [], []
train_macro_f1s, val_macro_f1s = [], []


print("starting training (optimizer: adamw, early stopping based on macro-f1 on 13 pathologies, excluding no finding)")
for epoch in range(1, config['train']['num_epochs'] + 1):
    # training phase
    model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []
    for images, labels in tqdm(train_loader, desc=f"epoch {epoch}/{config['train']['num_epochs']} (train)"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['train']['grad_clip_norm'])
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        
        probs = torch.sigmoid(logits)
        train_preds.append(probs.detach().cpu().numpy())
        train_labels.append(labels.detach().cpu().numpy())
    train_loss /= len(train_loader.dataset)
    train_losses.append(train_loss)
    
    # compute train macro-f1 on 13 pathologies (excluding no finding)
    train_preds = np.vstack(train_preds)
    train_labels = np.vstack(train_labels)
    train_preds_binary = (train_preds > 0.5).astype(int)
    train_preds_other = train_preds_binary[:, other_indices]
    train_labels_other = train_labels[:, other_indices]
    train_macro_f1 = f1_score(train_labels_other, train_preds_other, average='macro', zero_division=0)
    train_macro_f1s.append(train_macro_f1)
    
    # validation phase
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(dev_loader, desc=f"epoch {epoch}/{config['train']['num_epochs']} (val)"):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits)
            val_preds.append(probs.cpu().numpy())
            val_labels.append(labels.cpu().numpy())
    val_loss /= len(dev_loader.dataset)
    val_losses.append(val_loss)
    
    # compute val macro-f1 on 13 pathologies (excluding no finding)
    val_preds = np.vstack(val_preds)
    val_labels = np.vstack(val_labels)
    val_preds_binary = (val_preds > 0.5).astype(int)
    val_preds_other = val_preds_binary[:, other_indices]
    val_labels_other = val_labels[:, other_indices]
    val_macro_f1 = f1_score(val_labels_other, val_preds_other, average='macro', zero_division=0)
    val_macro_f1s.append(val_macro_f1)
    
    print(f"epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, train_macro_f1={train_macro_f1:.4f}, val_macro_f1={val_macro_f1:.4f}")
    
    # scheduler step based on val loss
    scheduler.step(val_loss)
    
    # early stopping & checkpoint based on val macro-f1 (excluding no finding)
    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        early_stop_counter = 0
        best_model_path = os.path.join(config['output']['checkpoint_dir'], config['output']['best_model_filename'])
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_macro_f1': best_val_macro_f1,
        }, best_model_path)
        print(f"  -> new best model saved (val_macro_f1={val_macro_f1:.4f})")
    else:
        early_stop_counter += 1
        if early_stop_counter >= config['train']['early_stopping_patience']:
            print(f"early stopping triggered at epoch {epoch} (no improvement for {config['train']['early_stopping_patience']} epochs)")
            break

print(f"\ntraining finished. best validation macro-f1 (13 pathologies, excluding no finding): {best_val_macro_f1:.4f}")

starting training (optimizer: adamw, early stopping based on macro-f1 on 13 pathologies, excluding no finding)


epoch 1/30 (train):   0%|          | 56/23692 [00:28<3:17:52,  1.99it/s]


KeyboardInterrupt: 

## 5. Model Evaluation

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='train loss')
plt.plot(val_losses, label='val loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.title('training & validation loss (bce)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_macro_f1s, label='train macro-f1 (13 pathologies)')
plt.plot(val_macro_f1s, label='val macro-f1 (13 pathologies)')
plt.xlabel('epoch')
plt.ylabel('macro-f1')
plt.legend()
plt.title('training & validation macro-f1 (excluding no finding)')
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(config['output']['plot_dir'], 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_model_path = os.path.join(config['output']['checkpoint_dir'], config['output']['best_model_filename'])
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="evaluating test set"):
        images = images.to(device)
        logits = model(images)
        probs = torch.sigmoid(logits)
        test_preds.append(probs.cpu().numpy())
        test_labels.append(labels.cpu().numpy())

test_preds = np.vstack(test_preds)
test_labels = np.vstack(test_labels)
test_preds_binary = (test_preds > 0.5).astype(int)


print("test set performance (threshold=0.5)")
print(classification_report(test_labels, test_preds_binary, target_names=label_cols, zero_division=0))

test_bal_acc = balanced_accuracy_score(test_labels, test_preds_binary)
print(f"\nbalanced accuracy on test set: {test_bal_acc:.4f}")

## 6. Threshold Tuning

In [ ]:
dev_probs, dev_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in tqdm(dev_loader, desc="extracting dev probabilities"):
        images = images.to(device)
        logits = model(images)
        probs = torch.sigmoid(logits)
        dev_probs.append(probs.cpu().numpy())
        dev_labels.append(labels.cpu().numpy())

dev_probs = np.vstack(dev_probs)
dev_labels = np.vstack(dev_labels)

thresholds = np.arange(0.1, 0.91, 0.05)
best_th = 0.5
best_macro_f1 = 0.0

print("threshold tuning (macro f1 on dev set, excluding no finding)")
for th in thresholds:
    preds_th = (dev_probs[:, other_indices] >= th).astype(int)
    labels_th = dev_labels[:, other_indices]
    macro_f1 = f1_score(labels_th, preds_th, average='macro', zero_division=0)
    print(f"threshold={th:.2f}: macro_f1={macro_f1:.4f}")
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_th = th

print(f"\nbest threshold: {best_th:.2f} (macro f1 on dev: {best_macro_f1:.4f})")

# save best threshold
with open(os.path.join(config['output']['tuning_dir'], 'best_threshold.txt'), 'w') as f:
    f.write(str(best_th))

# save dev probabilities for future use
np.save(os.path.join(config['output']['tuning_dir'], 'dev_probs.npy'), dev_probs)
np.save(os.path.join(config['output']['tuning_dir'], 'dev_labels.npy'), dev_labels)

In [ ]:
truly_normal_mask = df_dev['No Finding_label'] == 1.0
truly_normal_probs = dev_probs[truly_normal_mask]

correctly_flagged_normal = 0
total_normal = len(truly_normal_probs)

for row_probs in truly_normal_probs:
    positive_others = [i for i in other_indices if row_probs[i] >= best_th]
    if len(positive_others) == 0:
        correctly_flagged_normal += 1

normal_detection_rate = correctly_flagged_normal / total_normal if total_normal > 0 else 0.0


print("validation on genuinely normal cases (dev set):")
print(f"total genuinely normal samples in dev: {total_normal}")
print(f"correctly flagged as 'no significant findings': {correctly_flagged_normal}")
print(f"normal detection rate: {normal_detection_rate:.2%}")

# abnormal recall rate (sanity check untuk memastikan threshold tidak terlalu tinggi)
abnormal_mask = (df_dev[label_cols].drop(columns=['No Finding_label']).sum(axis=1) > 0)
abnormal_probs = dev_probs[abnormal_mask]
abnormal_detected = 0
total_abnormal = len(abnormal_probs)

for row_probs in abnormal_probs:
    positive_others = [i for i in other_indices if row_probs[i] >= best_th]
    if len(positive_others) > 0:
        abnormal_detected += 1

abnormal_recall_rate = abnormal_detected / total_abnormal if total_abnormal > 0 else 0.0
print(f"total genuinely abnormal samples in dev: {total_abnormal}")
print(f"correctly flagged with at least one pathology: {abnormal_detected}")
print(f"abnormal recall rate: {abnormal_recall_rate:.2%}")

print("\ninterpretation:")
if normal_detection_rate < 0.90:
    print("  -> warning: threshold too low. many normal cases are being over-called as abnormal.")
elif normal_detection_rate > 0.99 and abnormal_recall_rate < 0.50:
    print("  -> warning: threshold too high. many abnormal cases are being missed.")
else:
    print("  -> threshold balanced well between preserving normal cases and detecting abnormalities.")

In [ ]:
# evaluate on test set with optimal threshold
test_preds_optimal = (test_preds[:, other_indices] >= best_th).astype(int)
test_labels_optimal = test_labels[:, other_indices]

print(f"test set performance (optimal threshold={best_th:.2f})")
print(classification_report(test_labels_optimal, test_preds_optimal, target_names=[label_cols[i] for i in other_indices], zero_division=0))

test_bal_acc_optimal = balanced_accuracy_score(test_labels_optimal, test_preds_optimal)
print(f"\nbalanced accuracy on test set (optimal threshold): {test_bal_acc_optimal:.4f}")

## 7. Summary of Stage-B

In [ ]:
summary = f"""
====================================================================
stage-b training summary
====================================================================
experiment: {config['experiment']}
stage: {config['stage']}
device: {device}

dataset:
  - train: {len(df_train)} samples
  - dev: {len(df_dev)} samples
  - test: {len(df_test)} samples

model:
  - backbone: biovil-t
  - vision dimension: {config['vision_dim']}
  - freeze backbone: {config['model']['freeze_backbone']}
  - hidden dim: {config['model']['hidden_dim']}
  - dropout: {config['model']['dropout']}
  - total params: {total_params/1e6:.2f}m
  - trainable params: {trainable_params/1e6:.2f}m

training:
  - batch size: {config['train']['batch_size']}
  - learning rate: {config['train']['lr']}
  - weight decay: {config['train']['weight_decay']}
  - epochs: {len(train_losses)}
  - best validation macro-f1 (13 pathologies, excluding no finding): {best_val_macro_f1:.4f}

threshold tuning (on dev set, excluding no finding):
  - best threshold: {best_th:.2f}
  - macro f1 at best threshold: {best_macro_f1:.4f}

normal detection rate (dev set, genuinely normal cases):
  - correctly flagged as normal: {correctly_flagged_normal}/{total_normal}
  - normal detection rate: {normal_detection_rate:.2%}
  - abnormal recall rate: {abnormal_recall_rate:.2%}

test performance:
  - bal_acc (threshold=0.5): {test_bal_acc:.4f}
  - bal_acc (optimal threshold={best_th:.2f}): {test_bal_acc_optimal:.4f}

output directory: {config['output']['checkpoint_dir']}
====================================================================
"""

print(summary)

with open(os.path.join(config['output']['log_dir'], 'training_summary.txt'), 'w') as f:
    f.write(summary)

print("\nall done. stage-b training and evaluation complete.")